In [3]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data (from Week 4)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/molecular-biology/promoter-gene-sequences/promoters.data"
df = pd.read_csv(url, header=None, names=['label', 'id', 'sequence'])

# Clean sequences
def clean_sequence(seq):
    return seq.strip().upper()

df['sequence'] = df['sequence'].apply(clean_sequence)
df['label'] = (df['label'] == '+').astype(float)

# One-hot encode
def one_hot_encode(seq):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = torch.zeros(len(seq), 4)
    for i, nucleotide in enumerate(seq):
        if nucleotide in mapping:
            encoded[i, mapping[nucleotide]] = 1.0
        # Unknown nucleotides remain as zeros
    return encoded

# Encode all sequences
sequences = [one_hot_encode(seq) for seq in df['sequence']]
sequences = torch.stack(sequences)  # (106, 57, 4)
labels = torch.tensor(df['label'].values, dtype=torch.float32).unsqueeze(1)  # (106, 1)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Sequence shape: {X_train[0].shape}")  # (57, 4)

Training samples: 84
Test samples: 22
Sequence shape: torch.Size([57, 4])


In [2]:
# Load pre-trained model
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "InstaDeepAI/nucleotide-transformer-500m-human-ref"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,  # Promoter vs non-promoter
    problem_type="single_label_classification"
)

# Tokenize your sequences
sequences = df['sequence'].tolist()
tokens = tokenizer(sequences, padding=True, truncation=True, return_tensors="pt")



c:\Users\kudakwasheN\OneDrive - TC Recoveries\Documents\Personal\ProjectButterfly\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 390/390 [00:00<00:00, 26287.70it/s]
EsmForSequenceClassification LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING  

NameError: name 'df' is not defined

In [7]:
from torch.utils.data import DataLoader, Dataset
class PromoterDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item
    
dataset = PromoterDataset(tokens, torch.tensor(df['label'].values, dtype=torch.int64))

from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(range(len(dataset)), test_size=0.2, random_state=42, stratify=df['label'])
train_dataset = torch.utils.data.Subset(dataset, train_idx)
test_dataset = torch.utils.data.Subset(dataset, test_idx)

# Fine-tune on your 106 samples
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./promoter_model",
    num_train_epochs=10,
    per_device_train_batch_size=8,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

c:\Users\kudakwasheN\OneDrive - TC Recoveries\Documents\Personal\ProjectButterfly\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


KeyboardInterrupt: 